#### qLoRA Finetuning

In [8]:
!pip uninstall -y torchaudio
!pip install -q -U torch transformers datasets peft bitsandbytes accelerate evaluate rouge_score sacrebleu

In [9]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Configuration & Model Setup (Optimized for T4 15GB VRAM)

In [10]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 256

# 2. 4-bit Quantization Config (Fits larger models into T4's 16GB VRAM)

In [11]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16, # T4 supports fp16, not native bf16
#     bnb_4bit_use_double_quant=True
# )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # T4 GPUs natively support float16 (not bf16)
    bnb_4bit_use_double_quant=True,
)

In [12]:
bnb_config

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# 3. Adaptive Fine-Tuning via LoRA (PEFT)

In [15]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


lora_config

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.22.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'k_proj', 'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, astra_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)

In [16]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


# 4. Load & Split Dataset (Using a small slice for quick Colab execution)

In [17]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:600]")
dataset = dataset.train_test_split(test_size=0.15, seed=42)
train_data, val_data = dataset["train"], dataset["test"]

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [19]:
import pandas as pd
train_data.to_pandas()

,instruction,context,response,category
0,Classify each of the following as either title...,,J.K. Rowling: Harry Potter and the Sorcerer's ...,classification
1,Write a short story about a person who discove...,,It was going to be a long day. With her broth...,creative_writing
2,What do you know about the city of Aberdeen in...,,Aberdeen is a city located in the North East o...,open_qa
3,What is Bollywood,,"Bollywood is a movie industry based in Mumbai,...",open_qa
4,What is the appeal of Flathead Lake as a Summe...,,Flathead Lake is the largest natural freshwate...,creative_writing
...,...,...,...,...
505,What mammal lays the largest eggs?,,"The echidna, an Australian monotreme, is the l...",general_qa
506,Monitor and Merrimac refer to what in the pass...,"Merrimac Butte is a 5,627-foot (1,715-metre) s...",Monitor and Merrimac are two buttes which sit ...,summarization
507,What are common swimming strokes?,,The most common swimming strokes include the f...,brainstorming
508,Who were Jaehaerys I Targaryen's children?,,"Prince Aegon, Princess Daenerys, Prince Aemon,...",open_qa


In [20]:
def format_prompt(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}{tokenizer.eos_token}"
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

In [21]:
tokenized_train = train_data.map(format_prompt, remove_columns=train_data.column_names)
tokenized_val = val_data.map(format_prompt, remove_columns=val_data.column_names)

tokenized_train.to_pandas()

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

,input_ids,attention_mask,labels
0,"[1, 835, 2799, 4080, 29901, 13, 2385, 1598, 12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 2385, 1598, 12..."
1,"[1, 835, 2799, 4080, 29901, 13, 6113, 263, 327...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 6113, 263, 327..."
2,"[1, 835, 2799, 4080, 29901, 13, 5618, 437, 366...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 5618, 437, 366..."
3,"[1, 835, 2799, 4080, 29901, 13, 5618, 338, 892...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 5618, 338, 892..."
4,"[1, 835, 2799, 4080, 29901, 13, 5618, 338, 278...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 5618, 338, 278..."
...,...,...,...
505,"[1, 835, 2799, 4080, 29901, 13, 5618, 286, 485...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 5618, 286, 485..."
506,"[1, 835, 2799, 4080, 29901, 13, 7185, 2105, 32...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 7185, 2105, 32..."
507,"[1, 835, 2799, 4080, 29901, 13, 5618, 526, 361...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 5618, 526, 361..."
508,"[1, 835, 2799, 4080, 29901, 13, 22110, 892, 43...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 835, 2799, 4080, 29901, 13, 22110, 892, 43..."


# 5. Training Arguments (Tracking Train & Val Loss per Epoch)

In [22]:
training_args = TrainingArguments(
    output_dir="./adaptive_finetuned_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",        # Evaluate at the end of every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)


training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_checkpointing=False,
activation_checkpointing_kwargs=None,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulatio

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8)
)

In [25]:
trainer

# 6. Train the Model

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.841590,0.799886
2,0.837873,0.756827
3,0.828693,0.753052


TrainOutput(global_step=96, training_loss=1.3733231723308563, metrics={'train_runtime': 306.8689, 'train_samples_per_second': 4.986, 'train_steps_per_second': 0.313, 'total_flos': 2441775383838720.0, 'train_loss': 1.3733231723308563, 'epoch': 3.0})

# 6. Check Overfitting & Underfitting

In [27]:
print("\n--- Overfitting / Underfitting Diagnosis ---")
history = trainer.state.log_history
train_losses = [x["loss"] for x in history if "loss" in x]
eval_losses = [x["eval_loss"] for x in history if "eval_loss" in x]


--- Overfitting / Underfitting Diagnosis ---


In [30]:
trainer.state.log_history

[{'loss': 6.061186218261719,
  'grad_norm': 5.662792682647705,
  'learning_rate': 0.00018125000000000001,
  'epoch': 0.3125,
  'step': 10},
 {'loss': 0.9518942832946777,
  'grad_norm': 0.4497574269771576,
  'learning_rate': 0.00016041666666666667,
  'epoch': 0.625,
  'step': 20},
 {'loss': 0.8415904998779297,
  'grad_norm': 0.3221977651119232,
  'learning_rate': 0.00013958333333333333,
  'epoch': 0.9375,
  'step': 30},
 {'eval_loss': 0.7998862266540527,
  'eval_runtime': 4.5445,
  'eval_samples_per_second': 19.804,
  'eval_steps_per_second': 5.061,
  'epoch': 1.0,
  'step': 32},
 {'loss': 0.8697064399719239,
  'grad_norm': 0.32655954360961914,
  'learning_rate': 0.00011875,
  'epoch': 1.25,
  'step': 40},
 {'loss': 0.8282228469848633,
  'grad_norm': 0.23705798387527466,
  'learning_rate': 9.791666666666667e-05,
  'epoch': 1.5625,
  'step': 50},
 {'loss': 0.8378732681274415,
  'grad_norm': 0.2850325405597687,
  'learning_rate': 7.708333333333334e-05,
  'epoch': 1.875,
  'step': 60},
 {'

In [31]:
train_losses

[6.061186218261719,
 0.9518942832946777,
 0.8415904998779297,
 0.8697064399719239,
 0.8282228469848633,
 0.8378732681274415,
 0.7068975448608399,
 0.7857707023620606,
 0.8286925315856933]

In [32]:
eval_losses

[0.7998862266540527, 0.7568274736404419, 0.7530515193939209]

In [33]:
if train_losses and eval_losses:
    final_train_loss = train_losses[-1]
    final_val_loss = eval_losses[-1]
    print(f"Final Training Loss:   {final_train_loss:.4f}")
    print(f"Final Validation Loss: {final_val_loss:.4f}")

    if final_val_loss > final_train_loss * 1.25:
        print("Status: OVERFITTING detected (Validation loss is significantly higher than training loss).")
        print("Fix: Increase lora_dropout, reduce epochs, or add more training data.")
    elif final_train_loss > 1.8 and final_val_loss > 1.8:
        print("Status: UNDERFITTING detected (Both losses remain high).")
        print("Fix: Increase epochs, raise learning_rate, or increase LoRA rank (r=32).")
    else:
        print("Status: GOOD FIT (Training and validation losses are converging well).")

Final Training Loss:   0.8287
Final Validation Loss: 0.7531
Status: GOOD FIT (Training and validation losses are converging well).


# 7. Compute ROUGE and BLEU Scores on Validation Samples

In [34]:
print("\n--- Generating Predictions for ROUGE & BLEU ---")
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

predictions, references = [], []
model.eval()

# Evaluate on 20 validation samples to keep Colab inference fast
for sample in val_data.select(range(20)):
    prompt = f"### Instruction:\n{sample['instruction']}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    predictions.append(generated_text.strip())
    references.append(sample["response"].strip())

rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])

print(f"BLEU Score:    {bleu_results['bleu']:.4f}")
print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")


--- Generating Predictions for ROUGE & BLEU ---


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

BLEU Score:    0.0277
ROUGE-1 Score: 0.3110
ROUGE-2 Score: 0.1068
ROUGE-L Score: 0.2392
